In [ ]:
import os
import sys
import random
import argparse
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from timm.layers import trunc_normal_
import json

from collections import defaultdict
# Use a cross-platform font for the figures
plt.rcParams["font.family"] = ["DejaVu Sans"]
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ------------------------------
# Helper functions
# ------------------------------
def extract_slice_number(sample_id):
    """Return the numeric slice index embedded in a sample id."""
    try:
        return int(sample_id.split('_slice_')[1])
    except (IndexError, ValueError):
        return 0


def set_seed(seed):
    """Set all random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🌱 Global random seed set: {seed}")


In [ ]:
# ------------------------------
# 1. Loss functions
# ------------------------------
def build_loss(args, class_weights=None, device='cpu'):
    """Build the auxiliary classification loss; the CRF loss is computed inside StratigraphyCRFNet."""
    weight = None

    if class_weights is not None:
        weight = torch.as_tensor(
            class_weights,
            dtype=torch.float32,
            device=device
        )

    if args.loss == "cross_entropy":
        return nn.CrossEntropyLoss(
            weight=weight,
            label_smoothing=args.label_smoothing
        )

    if args.loss == "focal":
        return FocalLoss(
            gamma=args.focal_gamma,
            class_weights=weight
        )

    raise ValueError(f"Unsupported loss type: {args.loss}")


class FocalLoss(nn.Module):
    def __init__(
        self,
        gamma=2.0,
        class_weights=None,
        reduction="mean"
    ):
        super().__init__()

        self.gamma = gamma
        self.reduction = reduction

        # Register as a buffer so it follows the model/device automatically
        if class_weights is None:
            self.register_buffer("class_weights", None)
        else:
            self.register_buffer(
                "class_weights",
                class_weights.detach().clone()
            )

    def forward(self, logits, target):
        """logits: [N, num_classes], target: [N]"""
        log_probs = F.log_softmax(logits, dim=-1)

        log_pt = log_probs.gather(
            dim=1,
            index=target.unsqueeze(1)
        ).squeeze(1)

        pt = log_pt.exp()

        loss = -((1.0 - pt) ** self.gamma) * log_pt

        if self.class_weights is not None:
            loss = loss * self.class_weights[target]

        if self.reduction == "sum":
            return loss.sum()

        if self.reduction == "none":
            return loss

        return loss.mean()


In [ ]:
# ------------------------------
# 2. Data preprocessing
# ------------------------------
def fill_nan_value(train_set, val_set, test_set):
    def to_numeric(arr):
        if arr is None:
            return None
        arr = np.array(arr, dtype=np.float64)
        arr[np.isinf(arr)] = np.nan
        return arr

    train_set = to_numeric(train_set)
    val_set = to_numeric(val_set)
    test_set = to_numeric(test_set)

    col_means = None
    if train_set is not None:
        col_means = np.nanmean(train_set, axis=0)
        col_means[np.isnan(col_means)] = 0.0
        ind_train = np.where(np.isnan(train_set))
        train_set[ind_train] = np.take(col_means, ind_train[1])

    if val_set is not None and col_means is not None:
        ind_val = np.where(np.isnan(val_set))
        val_set[ind_val] = np.take(col_means, ind_val[1])
    if test_set is not None and col_means is not None:
        ind_test = np.where(np.isnan(test_set))
        test_set[ind_test] = np.take(col_means, ind_test[1])

    return train_set, val_set, test_set


def normalize_train_val_test(train, val, test):
    if train is None:
        return val, test

    min_val = np.min(train, axis=0)
    max_val = np.max(train, axis=0)
    range_val = max_val - min_val
    range_val[range_val == 0] = 1.0  

    train_norm = (train - min_val) / range_val
    val_norm = (val - min_val) / range_val if val is not None else None
    test_norm = (test - min_val) / range_val if test is not None else None

    return train_norm, val_norm, test_norm


In [ ]:
# ------------------------------
# 3. Custom dataset
# ------------------------------
class LogDataset(Dataset):
    def __init__(self, data, labels, well_names, sample_ids):
        self.data = data
        self.labels = labels
        self.well_names = well_names
        self.sample_ids = sample_ids

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx], self.well_names[idx], self.sample_ids[idx]


In [ ]:
# ------------------------------
# 4. Data loading: fixed test wells + random 8:2 split of remaining wells
# ------------------------------
# Default 17-channel feature set used by the model. It matches the synthetic
# dataset shipped in `data/synthetic` and can be overridden with --feature_cols.
DEFAULT_FEATURE_COLS = [
    'GR_INPEFA',
    'GR', 'CNL', 'AC', 'CAL', 'DEN', 'GR_A2', 'GR_D1', 'DEN_A1',
    'CNL_A3', 'CNL_D3', 'AC_A1',
    'vector_dim_1', 'vector_dim_2', 'vector_dim_3', 'vector_dim_4', 'vector_dim_5'
]


def load_single_dir(data_dir, seq_length, feature_cols,
                    sample_id_col='sample_id',
                    label_col='formation_label',
                    file_exts=('.csv', '.xlsx', '.xls')):
    """Load every well file in `data_dir` into per-well slice sequences."""
    well_data_list = []
    all_labels = []

    well_files = sorted(
        f for f in os.listdir(data_dir)
        if f.lower().endswith(tuple(file_exts))
    )
    if not well_files:
        raise FileNotFoundError(
            f"No well files ({', '.join(file_exts)}) found in {data_dir}"
        )

    for file in well_files:
        well_name = os.path.splitext(file)[0]
        file_path = os.path.join(data_dir, file)
        try:
            if file.lower().endswith('.csv'):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)
            print(f"Loading well {well_name} (file: {file}, rows: {len(df)})")

            missing_cols = [col for col in feature_cols if col not in df.columns]
            if missing_cols:
                raise KeyError(f"Missing feature columns in {file}: {missing_cols}")
            if sample_id_col not in df.columns:
                raise KeyError(
                    f"Missing sample id column '{sample_id_col}' in {file}"
                )
            if label_col not in df.columns:
                raise KeyError(f"Missing label column '{label_col}' in {file}")

            for col in feature_cols:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            well_sequences = []
            well_labels = []
            well_names = []
            sample_ids = []

            sample_id_groups = df.groupby(sample_id_col)

            sorted_sample_ids = sorted(
                sample_id_groups.groups.keys(),
                key=extract_slice_number
            )

            for sample_id in sorted_sample_ids:
                group = sample_id_groups.get_group(sample_id)
                features = group[feature_cols].values
                if len(features) == seq_length:
                    well_sequences.append(features)
                    label = group[label_col].iloc[0]
                    well_labels.append(label)
                    well_names.append(well_name)
                    sample_ids.append(sample_id)
                else:
                    print(
                        f"Skip {well_name}/{sample_id}: {len(features)} rows, "
                        f"expected {seq_length}"
                    )

            if len(well_sequences) == 0:
                print(f"No valid sequence in well {well_name}; skipped")
                continue

            well_data_list.append({
                'well_name': well_name,
                'sequences': well_sequences,
                'labels': well_labels,
                'well_names': well_names,
                'sample_ids': sample_ids,
                'sample_count': len(well_sequences)
            })
            all_labels.extend(well_labels)

            print(f"Well {well_name}: {len(sample_ids)} slices loaded")

        except Exception as e:
            print(f"Failed to load well {well_name}: {e}; skipped")
            continue

    if not well_data_list:
        raise ValueError(f"No valid well data could be loaded from {data_dir}")

    return well_data_list, all_labels


def load_log_data_train_val_test(data_dir, target_test_wells, train_ratio=0.8,
                                 seq_length=120, random_seed=42,
                                 feature_cols=None,
                                 sample_id_col='sample_id',
                                 label_col='formation_label'):
    """
    Load data: the wells listed in `target_test_wells` form the test set and
    the remaining wells are split into training and validation sets by
    `train_ratio`.
    """
    if feature_cols is None:
        feature_cols = DEFAULT_FEATURE_COLS

    print(f"\n=== Loading all wells from {data_dir} ===")
    all_well_data, _ = load_single_dir(
        data_dir,
        seq_length,
        feature_cols,
        sample_id_col=sample_id_col,
        label_col=label_col
    )

    rng = random.Random(random_seed)

    test_well_data = []
    train_val_well_data = []
    target_test_wells = set(target_test_wells)

    for well in all_well_data:
        if well['well_name'] in target_test_wells:
            test_well_data.append(well)
        else:
            train_val_well_data.append(well)

    if len(test_well_data) == 0:
        raise ValueError(
            "None of the requested test wells were found; check --test_wells"
        )
    if len(train_val_well_data) < 2:
        raise ValueError(
            "At least two non-test wells are required to build a train/validation split"
        )

    rng.shuffle(train_val_well_data)
    train_count = int(len(train_val_well_data) * train_ratio)
    train_count = max(1, min(train_count, len(train_val_well_data) - 1))

    train_well_data = train_val_well_data[:train_count]
    val_well_data = train_val_well_data[train_count:]

    total_wells = len(all_well_data)
    print(
        f"\nWell split: total={total_wells}, "
        f"train={len(train_well_data)}, val={len(val_well_data)}, "
        f"test={len(test_well_data)}"
    )

    all_labels = []
    for well in train_well_data + val_well_data + test_well_data:
        all_labels.extend(well['labels'])

    label_counts = pd.Series(all_labels).value_counts().sort_index()
    print("\nLabel counts:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} slices")

    def merge_wells_for_crf(wells, seq_slices=5):
        all_seq_blocks = []
        all_label_blocks = []
        all_well_names_blocks = []
        all_sample_ids_blocks = []

        for well in wells:
            seqs = np.array(well['sequences']).transpose(0, 2, 1)  # [N, C, T]
            labels = np.array(well['labels'])
            well_names = np.array(well['well_names'])
            sample_ids = np.array(well['sample_ids'])

            for i in range(len(seqs) - seq_slices + 1):
                all_seq_blocks.append(seqs[i:i + seq_slices])        # [S, C, T]
                all_label_blocks.append(labels[i:i + seq_slices])    # [S]
                all_well_names_blocks.append(well_names[i:i + seq_slices])
                all_sample_ids_blocks.append(sample_ids[i:i + seq_slices])

        return (
            np.array(all_seq_blocks),
            np.array(all_label_blocks),
            np.array(all_well_names_blocks),
            np.array(all_sample_ids_blocks)
        )

    X_train, y_train, train_well_names, train_sample_ids = merge_wells_for_crf(train_well_data)
    X_val, y_val, val_well_names, val_sample_ids = merge_wells_for_crf(val_well_data)
    X_test, y_test, test_well_names, test_sample_ids = merge_wells_for_crf(test_well_data)

    train_well_names_list = [well['well_name'] for well in train_well_data]
    val_well_names_list = [well['well_name'] for well in val_well_data]
    test_well_names_list = [well['well_name'] for well in test_well_data]
    total_samples = len(X_train) + len(X_val) + len(X_test)

    print("\nDataset split (fixed test wells + random split of the rest):")
    print(f"  Total sequence blocks: {total_samples}")
    print(f"  Train: {len(train_well_data)} wells, {len(X_train)} blocks "
          f"({len(X_train) / total_samples * 100:.1f}%)")
    print(f"    wells: {train_well_names_list}")
    print(f"  Val: {len(val_well_data)} wells, {len(X_val)} blocks "
          f"({len(X_val) / total_samples * 100:.1f}%)")
    print(f"    wells: {val_well_names_list}")
    print(f"  Test: {len(test_well_data)} wells, {len(X_test)} blocks "
          f"({len(X_test) / total_samples * 100:.1f}%)")
    print(f"    wells: {test_well_names_list}")

    le = LabelEncoder()
    le.fit(np.concatenate([y_train.flatten(), y_val.flatten(), y_test.flatten()]))

    y_train_encoded = le.transform(y_train.flatten()).reshape(y_train.shape)
    y_val_encoded = le.transform(y_val.flatten()).reshape(y_val.shape)
    y_test_encoded = le.transform(y_test.flatten()).reshape(y_test.shape)

    num_classes = len(le.classes_)
    print("\nLabel encoding:")
    print(f"  num_classes={num_classes}, mapping={dict(zip(le.classes_, range(num_classes)))}")

    return (X_train, y_train_encoded, train_well_names, train_sample_ids,
            X_val, y_val_encoded, val_well_names, val_sample_ids,
            X_test, y_test_encoded, test_well_names, test_sample_ids,
            num_classes, le, label_counts, train_well_data, val_well_data, test_well_data, random_seed)


In [ ]:
# ------------------------------
# 5. Detailed evaluation (whole-well CRF decoding)
# ------------------------------
def detailed_evaluate_model(
    dataloader,
    model,
    aux_loss_fn,
    device,
    args,
    label_encoder,
    eval_epoch,
):
    model.eval()

    total_loss = 0.0
    total_crf_loss = 0.0
    total_aux_loss = 0.0
    total_moe_loss = 0.0
    total_tokens = 0

    # Store emission scores per unique slice
    emission_records = {}

    with torch.no_grad():
        for (
            x_seq,
            y_seq,
            well_names_seq,
            sample_ids_seq
        ) in dataloader:

            x_seq = x_seq.to(
                device,
                non_blocking=True
            )

            y_seq = y_seq.to(
                device,
                non_blocking=True
            )

            crf_loss, moe_loss, emissions = model(
                x_seq,
                labels=y_seq,
                num_epoch_i=eval_epoch,
                warm_up_epoch=args.warm_up_epoch,
                return_emissions=True
            )

            aux_loss = aux_loss_fn(
                emissions.reshape(
                    -1,
                    args.num_class
                ),
                y_seq.reshape(-1)
            )

            batch_loss = (
                crf_loss
                + args.aux_loss_rate * aux_loss
                + args.moe_loss_rate * moe_loss
            )

            num_tokens = y_seq.numel()

            total_loss += (
                batch_loss.item() * num_tokens
            )

            total_crf_loss += (
                crf_loss.item() * num_tokens
            )

            total_aux_loss += (
                aux_loss.item() * num_tokens
            )

            total_moe_loss += (
                moe_loss.item() * num_tokens
            )

            total_tokens += num_tokens

            # After default collate, metadata is laid out by sequence position
            sample_ids_by_batch = list(
                zip(*sample_ids_seq)
            )

            well_names_by_batch = list(
                zip(*well_names_seq)
            )

            # Accumulate emission scores for every unique slice
            for b in range(x_seq.size(0)):
                for s in range(x_seq.size(1)):
                    sample_id = (
                        sample_ids_by_batch[b][s]
                    )

                    well_name = (
                        well_names_by_batch[b][s]
                    )

                    true_label = int(
                        y_seq[b, s].item()
                    )

                    current_emission = (
                        emissions[b, s]
                        .detach()
                        .float()
                        .cpu()
                    )

                    if sample_id not in emission_records:
                        emission_records[sample_id] = {
                            "well_name": well_name,
                            "true": true_label,
                            "slice_number":
                                extract_slice_number(
                                    sample_id
                                ),
                            "emission_sum":
                                current_emission.clone(),
                            "count": 1
                        }

                    else:
                        record = emission_records[
                            sample_id
                        ]

                        # A slice must have the same label in all overlapping windows
                        if record["true"] != true_label:
                            raise ValueError(
                                f"Inconsistent label for the same sample: "
                                f"{sample_id}"
                            )

                        record[
                            "emission_sum"
                        ] += current_emission

                        record["count"] += 1

    # Group slices by well and decode the whole well
    well_to_samples = defaultdict(list)

    for sample_id, record in emission_records.items():
        well_to_samples[
            record["well_name"]
        ].append(
            (sample_id, record)
        )

    decoded_rows = []

    for well_name, well_items in (
        well_to_samples.items()
    ):
        # Sort strictly by slice number
        well_items = sorted(
            well_items,
            key=lambda item:
                item[1]["slice_number"]
        )

        # Average the emissions of a slice over overlapping windows
        averaged_emissions = torch.stack([
            record["emission_sum"]
            / float(record["count"])
            for _, record in well_items
        ])

        # [1, well_length, num_classes]
        whole_well_emissions = (
            averaged_emissions
            .unsqueeze(0)
            .to(device)
        )

        # Run CRF decoding once per well
        whole_well_predictions = model.decode(
            whole_well_emissions
        )[0].detach().cpu().tolist()

        for (
            sample_id,
            record
        ), pred_label in zip(
            well_items,
            whole_well_predictions
        ):
            decoded_rows.append({
                "sample_id": sample_id,
                "well_name": well_name,
                "true": record["true"],
                "pred": int(pred_label),
                "slice_number":
                    record["slice_number"]
            })

    df_clean = pd.DataFrame(decoded_rows)

    # Keep the final report order stable
    df_clean = df_clean.sort_values(
        ["well_name", "slice_number"]
    ).reset_index(drop=True)

    all_labels = df_clean[
        "true"
    ].to_numpy()

    all_preds = df_clean[
        "pred"
    ].to_numpy()

    avg_loss = (
        total_loss / max(total_tokens, 1)
    )

    avg_crf_loss = (
        total_crf_loss / max(total_tokens, 1)
    )

    avg_aux_loss = (
        total_aux_loss / max(total_tokens, 1)
    )

    avg_moe_loss = (
        total_moe_loss / max(total_tokens, 1)
    )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _
    ) = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _
    ) = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    class_report = classification_report(
        all_labels,
        all_preds,
        target_names=label_encoder.classes_,
        zero_division=0
    )

    conf_matrix = confusion_matrix(
        all_labels,
        all_preds
    )

    well_acc_dict = {}

    for well_name, group in df_clean.groupby(
        "well_name"
    ):
        sorted_group = group.sort_values(
            "slice_number"
        ).reset_index(drop=True)

        well_acc = accuracy_score(
            sorted_group["true"],
            sorted_group["pred"]
        )

        well_acc_dict[well_name] = {
            "accuracy": well_acc,
            "sample_count": len(sorted_group),
            "correct_count": (
                sorted_group["true"]
                == sorted_group["pred"]
            ).sum(),
            "incorrect_count": (
                sorted_group["true"]
                != sorted_group["pred"]
            ).sum(),
            "sorted_samples": sorted_group[[
                "sample_id",
                "true",
                "pred",
                "slice_number"
            ]]
        }

    return {
        "avg_loss": avg_loss,
        "avg_crf_loss": avg_crf_loss,
        "avg_aux_loss": avg_aux_loss,
        "avg_moe_loss": avg_moe_loss,

        "overall_accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "class_report": class_report,
        "confusion_matrix": conf_matrix,
        "well_accuracy": well_acc_dict,

        "raw_data": (
            df_clean["well_name"].values,
            all_labels,
            all_preds,
            df_clean["sample_id"].values
        )
    }


# Sequence error statistics
def count_sequence_errors(well_accuracy_dict, class_order):
    """
    Count predicted transitions that constitute sequence errors under the stratigraphic prior.

    Parameters
    ----------
    well_accuracy_dict : dict
        Per-well evaluation records produced by `detailed_evaluate_model`.
    class_order : list[int]
        Encoded class indices ordered from the shallowest (top) member to the
        deepest (bottom) member.

    A transition is a sequence error when it moves upward (rank decreases) or when
    it skips one or more members (rank increases by more than one).
    """
    order_rank = {tag: rank for rank, tag in enumerate(class_order)}

    reverse_count = 0
    skip_count = 0
    total_transition_count = 0

    sequence_error_details = []

    for well_name, well_info in well_accuracy_dict.items():
        samples = well_info["sorted_samples"].sort_values(
            "slice_number"
        ).reset_index(drop=True)

        predictions = samples["pred"].astype(int).tolist()
        sample_ids = samples["sample_id"].tolist()

        for index, (current_tag, next_tag) in enumerate(
            zip(predictions[:-1], predictions[1:])
        ):
            current_rank = order_rank[current_tag]
            next_rank = order_rank[next_tag]
            rank_difference = next_rank - current_rank

            total_transition_count += 1
            sequence_error_type = None

            if rank_difference < 0:
                reverse_count += 1
                sequence_error_type = "reverse_transition"
            elif rank_difference > 1:
                skip_count += 1
                sequence_error_type = "layer_skip"

            if sequence_error_type is not None:
                sequence_error_details.append({
                    "well_name": well_name,
                    "from_sample": sample_ids[index],
                    "to_sample": sample_ids[index + 1],
                    "from_tag": current_tag,
                    "to_tag": next_tag,
                    "type": sequence_error_type
                })

    return {
        "total_transitions": total_transition_count,
        "reverse_transitions": reverse_count,
        "skip_transitions": skip_count,
        "total_sequence_errors": reverse_count + skip_count,
        "sequence_error_rate": (
            (reverse_count + skip_count) / total_transition_count
            if total_transition_count > 0 else 0.0
        ),
        "details": sequence_error_details
    }


In [ ]:
# ------------------------------
# 6. Result saving
# ------------------------------
def save_cls_result(args, eval_results, train_time, label_encoder, save_path, random_seed):
    """Save the complete training result to a CSV file."""
    result_df = pd.DataFrame({
        'timestamp': [time.strftime('%Y-%m-%d %H:%M:%S')],
        'random_seed': [random_seed],
        'data_dir': [args.data_dir],
        'seq_length': [args.seq_length],
        'emb_dim': [args.emb_dim],
        'depth': [args.depth],
        'sparse_rate': [args.sparse_rate],
        'shapelet_window': [args.shape_size],
        'shapelet_stride': [args.shape_stride],
        'moe_num_experts': [args.moe_num_experts],
        'MoE Top-k': [args.moe_top_k],
        'warm_up_epoch': [args.warm_up_epoch],
        'sparse_ramp_epoch': [args.sparse_ramp_epoch],
        'transformer_layers': [args.transformer_layers],
        'transformer_nhead': [args.transformer_nhead],
        'lr': [args.lr],
        'weight_decay': [args.weight_decay],
        'batch_size': [args.batch_size],
        'epoch': [args.epoch],
        'overall_accuracy': [eval_results['overall_accuracy']],
        'precision_macro': [eval_results['precision_macro']],
        'recall_macro': [eval_results['recall_macro']],
        'f1_macro': [eval_results['f1_macro']],
        'precision_micro': [eval_results['precision_micro']],
        'recall_micro': [eval_results['recall_micro']],
        'f1_micro': [eval_results['f1_micro']],
        'train_time_minutes': [train_time / 60],
        'loss': [args.loss],
        'optimizer': [args.optimizer],
        'moe_loss_rate': [args.moe_loss_rate]
    })

    if os.path.exists(save_path):
        result_df.to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        result_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"📁 Training results saved to: {save_path}")


In [ ]:
# ------------------------------
# 7. Main training entry point
# ------------------------------
if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description="SoftShapeTransformerNet (three-path hybrid model) - well-log stratigraphy classification"
    )

    # Basic settings
    parser.add_argument('--random_seed', type=int, default=0, help='Random seed (default: 0)')
    parser.add_argument('--data_dir', type=str, default='./data/synthetic',
                        help='Directory with one well file per well (.csv/.xlsx)')
    parser.add_argument('--seq_length', type=int, default=120,
                        help='Fixed depth-sample length of one slice')
    parser.add_argument('--train_ratio', type=float, default=0.8,
                        help='Fraction of the non-test wells used for training (default: 0.8)')
    parser.add_argument('--test_wells', type=str, default='SYN_TEST_01,SYN_TEST_02',
                        help='Comma-separated well names held out as the test set')
    parser.add_argument('--stratum_order', type=str, default='J5,J4,J3,J2,J1',
                        help='Stratigraphic members ordered from top (shallow) to bottom (deep)')
    parser.add_argument('--sample_id_col', type=str, default='sample_id',
                        help='Column holding the slice identifier')
    parser.add_argument('--label_col', type=str, default='formation_label',
                        help='Column holding the slice label')
    parser.add_argument('--feature_cols', type=str, default='',
                        help='Comma-separated feature columns; empty uses the default 17 channels')

    # Model settings
    parser.add_argument('--emb_dim', type=int, default=128, help='Shapelet embedding dimension')
    parser.add_argument('--depth', type=int, default=2, help='Number of SoftShapeNet layers')
    parser.add_argument('--sparse_rate', type=float, default=0.3, help='Final target sparsity rate')
    parser.add_argument('--shape_size', type=int, default=60, help='Shapelet window size')
    parser.add_argument('--shape_stride', type=int, default=5, help='Shapelet sliding stride')
    parser.add_argument('--moe_num_experts', type=int, default=4, help='Number of MoE experts')
    parser.add_argument('--moe_top_k', type=int, default=2, help='Experts selected per token')
    parser.add_argument('--transformer_nhead', type=int, default=4,
                        help='Number of attention heads')
    parser.add_argument('--transformer_layers', type=int, default=2,
                        help='Number of Transformer encoder layers')

    # Training settings
    parser.add_argument('--warm_up_epoch', type=int, default=5,
                        help='Epochs without sparsification')
    parser.add_argument('--sparse_ramp_epoch', type=int, default=20,
                        help='Epochs used to ramp the sparsity from 0 to the target')
    parser.add_argument('--moe_loss_rate', type=float, default=0.02,
                        help='Weight of the differentiable MoE balance loss')
    parser.add_argument('--loss', type=str, default='cross_entropy',
                        help='Auxiliary loss: cross_entropy or focal')
    parser.add_argument('--optimizer', type=str, default='adam', help='Optimizer: adam or sgd')
    parser.add_argument('--lr', type=float, default=0.001, help='Initial learning rate')
    parser.add_argument('--weight_decay', type=float, default=1e-5, help='Weight decay')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
    parser.add_argument('--epoch', type=int, default=200, help='Number of training epochs')
    parser.add_argument('--cuda', type=str, default='cuda:0',
                        help='Device string (cuda:0 or cpu)')

    parser.add_argument('--aux_loss_rate', type=float, default=0.3,
                        help='Weight of the auxiliary CE/Focal loss')
    parser.add_argument('--label_smoothing', type=float, default=0.05,
                        help='Label smoothing for cross-entropy')
    parser.add_argument('--focal_gamma', type=float, default=2.0,
                        help='Gamma parameter of Focal Loss')

    # Output settings
    parser.add_argument('--save_dir', type=str, default='./result_specified_split',
                        help='Directory for output files')
    parser.add_argument('--save_csv_name', type=str,
                        default='log_stratigraphy_random_split_result',
                        help='Result CSV file name')
    parser.add_argument('--save_model_name', type=str, default='best_model_random_split.pth',
                        help='Best model file name')

    # Options used when the notebook runs inside Jupyter. Edit this list to
    # change settings without leaving the notebook, e.g.
    # NOTEBOOK_ARGS = ['--epoch', '20', '--stratum_order', 'J5,J4,J3,J2,J1']
    NOTEBOOK_ARGS = []

    if 'ipykernel' in sys.modules:
        args = parser.parse_args(NOTEBOOK_ARGS)
    else:
        args, _ = parser.parse_known_args()

    # ============================================================
    # Fixed configuration: whole-well CRF, 4 experts with Top-k=2, progressive sparsity 0.3
    # ============================================================
    args.experiment_name = "MoE4_Top2_Sparse03"
    args.transition_penalty = -3.0

    args.save_model_name = (
        f"best_model_{args.experiment_name}_"
        f"seed_{args.random_seed}.pth"
    )
    args.save_csv_name = (
        f"log_stratigraphy_{args.experiment_name}_"
        f"seed_{args.random_seed}"
    )

    print("=" * 60)
    print(f"Experiment: {args.experiment_name}")
    print(f"MoE experts: {args.moe_num_experts}")
    print(f"MoE Top-k: {args.moe_top_k}")
    print(f"Target sparsity: {args.sparse_rate}")
    print(f"Warm-up epochs: {args.warm_up_epoch}")
    print(f"Sparse ramp epochs: {args.sparse_ramp_epoch}")
    print(f"MoE loss weight: {args.moe_loss_rate}")
    print(f"Invalid transition penalty: {args.transition_penalty}")
    print("=" * 60)

    # Initialize environment
    os.makedirs(args.save_dir, exist_ok=True)
    device = torch.device(args.cuda if torch.cuda.is_available() else "cpu")
    print(f"=== Training environment setup ===")
    print(f"Device: {device} | Output directory: {args.save_dir} | Random seed: {args.random_seed}")
    print(f"Data directory: {args.data_dir}")
    print(f"Training ratio of remaining wells: {args.train_ratio} | Validation ratio: {1 - args.train_ratio}")

    # Load well data
    print("\n=== Loading well-log data ===")
    target_test_wells = [
        name.strip() for name in args.test_wells.split(',') if name.strip()
    ]
    if not target_test_wells:
        raise ValueError("--test_wells must contain at least one well name")
    print(f"Fixed test wells: {target_test_wells}")

    if args.feature_cols.strip():
        feature_cols = [
            col.strip() for col in args.feature_cols.split(',') if col.strip()
        ]
    else:
        feature_cols = DEFAULT_FEATURE_COLS
    print(f"Feature channels ({len(feature_cols)}): {feature_cols}")

    try:
        (X_train, y_train, train_well_names, train_sample_ids,
         X_val, y_val, val_well_names, val_sample_ids,
         X_test, y_test, test_well_names, test_sample_ids,
         num_classes, label_encoder, label_counts, train_wells, val_wells, test_wells, used_seed) = load_log_data_train_val_test(
            data_dir=args.data_dir,
            target_test_wells=target_test_wells,
            train_ratio=args.train_ratio,
            seq_length=args.seq_length,
            random_seed=args.random_seed,
            feature_cols=feature_cols,
            sample_id_col=args.sample_id_col,
            label_col=args.label_col
        )
    except Exception as e:
        print(f"❌ Data loading failed: {str(e)}")
        sys.exit(1)

    # Set the global random seed
    set_seed(args.random_seed)
    start_time = time.time()

    args.num_class = num_classes
    args.num_channels = X_train.shape[2]  # number of feature channels
    args.seq_len = X_train.shape[3]       # sequence length

    print(f"\n=== Data dimensions ===")
    print(f"Train: {X_train.shape[0]} sequences x {X_train.shape[1]} consecutive slices x {args.num_channels} channels x {args.seq_len} depth samples")
    print(f"Validation: {X_val.shape[0]} sequences x {X_val.shape[1]} consecutive slices x {args.num_channels} channels x {args.seq_len} depth samples")
    print(f"Test: {X_test.shape[0]} sequences x {X_test.shape[1]} consecutive slices x {args.num_channels} channels x {args.seq_len} depth samples")

    # Preprocessing: flatten to 3-D for imputation/normalization, then restore 4-D
    print("\n=== Data preprocessing ===")
    B_tr, S, C, T_len = X_train.shape
    B_va = X_val.shape[0]
    B_te = X_test.shape[0]

    X_train_3d = X_train.reshape(-1, C, T_len)
    X_val_3d = X_val.reshape(-1, C, T_len)
    X_test_3d = X_test.reshape(-1, C, T_len)

    X_train_3d, X_val_3d, X_test_3d = fill_nan_value(X_train_3d, X_val_3d, X_test_3d)
    X_train_3d, X_val_3d, X_test_3d = normalize_train_val_test(X_train_3d, X_val_3d, X_test_3d)

    X_train = X_train_3d.reshape(B_tr, S, C, T_len)
    X_val = X_val_3d.reshape(B_va, S, C, T_len)
    X_test = X_test_3d.reshape(B_te, S, C, T_len)

    print("✅ Missing-value imputation and min-max normalisation completed")

    flat_train_labels = y_train.reshape(-1)

    # Class weights follow the encoded class indices 0..num_classes-1
    class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=flat_train_labels)

    # Normalize weights to mean 1 to stabilize the auxiliary loss
    class_weights = class_weights / class_weights.mean()

    print("Class mapping:")
    print(dict(enumerate(label_encoder.classes_)))

    print(
        "⚖️ Class weights:",
        {
            label_encoder.classes_[i]: round(float(class_weights[i]), 4)
            for i in range(num_classes)
        }
    )

    # Convert the user-defined top-to-bottom member names into the encoded
    # class indices expected by the CRF prior.
    stratum_order = [
        name.strip() for name in args.stratum_order.split(',') if name.strip()
    ]

    label_class_names = [str(cls) for cls in label_encoder.classes_]

    if (
        len(stratum_order) != num_classes
        or set(stratum_order) != set(label_class_names)
    ):
        raise ValueError(
            "The names in --stratum_order must match the labels found in the data. "
            f"Provided order: {stratum_order}; data labels: {label_class_names}"
        )

    # Compare as strings so that numeric or string labels both work.
    name_to_index = {
        str(cls): int(idx) for idx, cls in enumerate(label_encoder.classes_)
    }
    class_order = [name_to_index[name] for name in stratum_order]

    print("Stratigraphic prior (top -> bottom):", " -> ".join(stratum_order))
    print(
        "Encoded class order:",
        {name: int(idx) for name, idx in zip(stratum_order, class_order)}
    )

    # Build datasets and dataloaders
    train_set = LogDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(y_train).long(),
        train_well_names.tolist(),
        train_sample_ids.tolist()
    )
    val_set = LogDataset(
        torch.from_numpy(X_val).float(),
        torch.from_numpy(y_val).long(),
        val_well_names.tolist(),
        val_sample_ids.tolist()
    )
    test_set = LogDataset(
        torch.from_numpy(X_test).float(),
        torch.from_numpy(y_test).long(),
        test_well_names.tolist(),
        test_sample_ids.tolist()
    )

    num_workers = 0 if sys.platform == 'win32' else 4
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, num_workers=num_workers,
                              pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=args.batch_size * 2, shuffle=False, num_workers=num_workers,
                            pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=args.batch_size * 2, shuffle=False, num_workers=num_workers,
                             pin_memory=True)
    print(f"\n=== Data loaders ===")
    print(f"Train iterations: {len(train_loader)} | Validation iterations: {len(val_loader)} | Test iterations: {len(test_loader)}")
    print(f"✅ shuffle disabled for the validation and test DataLoaders to preserve sample order")

    # Initialize the model
    print("\n=== Initialising model ===")
    try:
        from models.PICSD import SoftShapeTransformerNet, StratigraphyCRFNet
    except ImportError as e:
        print(f"❌ Model import failed: {str(e)}")
        print("Make sure PICSD.py is available in the models directory")
        sys.exit(1)

    base_model = SoftShapeTransformerNet(
        seq_len=args.seq_len,
        shape_size=args.shape_size,
        num_channels=args.num_channels,
        emb_dim=args.emb_dim,
        sparse_rate=args.sparse_rate,
        depth=args.depth,
        num_experts=args.moe_num_experts,
        num_classes=args.num_class,
        stride=args.shape_stride,
        transformer_nhead=args.transformer_nhead,
        transformer_layers=args.transformer_layers,
        moe_top_k=args.moe_top_k,
        sparse_ramp_epoch=args.sparse_ramp_epoch
    ).to(device)

    model = StratigraphyCRFNet(
        base_model=base_model,
        num_classes=args.num_class,
        transition_penalty=args.transition_penalty,
        class_order=class_order
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ Model initialised: {total_params / 1e6:.2f} M trainable parameters")
    print(f"📊 Architecture: three-path hybrid model (SoftShape MoE + class-specific Transformer + generic Transformer)")

    # Loss function and optimizer
    aux_loss_fn = build_loss(args, class_weights=class_weights, device=device)
    if args.optimizer == 'adam':
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    elif args.optimizer == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, weight_decay=args.weight_decay, momentum=0.9,
                                    nesterov=True)
    else:
        raise ValueError(f"Unsupported optimizer: {args.optimizer}")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epoch, eta_min=1e-6)

    import copy

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_f1 = -float("inf")
    best_val_acc = 0.0
    best_epoch = 0
    best_model_state = None
    early_stop_counter = 0
    patience = 30

    print("\n=== Training started ===")
    for epoch in range(args.epoch):
        model.train()
        train_loss = 0.0
        train_crf_loss = 0.0
        train_aux_loss = 0.0
        train_moe_loss = 0.0

        router_importance_sum = np.zeros(args.moe_num_experts, dtype=np.float64)
        router_selection_sum = np.zeros(args.moe_num_experts, dtype=np.float64)
        router_entropy_sum = 0.0
        router_stat_batches = 0
        model.base_model.moe.reset_routing_stats()

        correct = 0
        total = 0

        for x_seq, y_seq, _, _ in train_loader:
            x_seq = x_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            crf_loss, moe_loss, emissions = model(
                x_seq,
                labels=y_seq,
                num_epoch_i=epoch,
                warm_up_epoch=args.warm_up_epoch,
                return_emissions=True
            )

            aux_loss = aux_loss_fn(
                emissions.reshape(-1, args.num_class),
                y_seq.reshape(-1)
            )

            batch_loss = (
                crf_loss
                + args.aux_loss_rate * aux_loss
                + args.moe_loss_rate * moe_loss
            )

            if not torch.isfinite(batch_loss):
                raise RuntimeError(
                    f"Non-finite loss detected: "
                    f"total={batch_loss.item()}, "
                    f"crf={crf_loss.item()}, "
                    f"aux={aux_loss.item()}, "
                    f"moe={moe_loss.item()}"
                )

            batch_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            batch_size = x_seq.size(0)

            train_loss += batch_loss.item() * batch_size
            train_crf_loss += crf_loss.item() * batch_size
            train_aux_loss += aux_loss.item() * batch_size
            train_moe_loss += moe_loss.item() * batch_size

            moe_module = model.base_model.moe
            if moe_module.last_importance is not None:
                router_importance_sum += moe_module.last_importance.cpu().numpy()
                router_selection_sum += moe_module.last_selection_share.cpu().numpy()
                router_entropy_sum += float(moe_module.last_router_entropy.cpu().item())
                router_stat_batches += 1

            # Decode the emissions from this forward pass; no extra model call
            with torch.no_grad():
                best_paths = model.decode(emissions.detach())

                correct += (best_paths == y_seq).sum().item()
                total += y_seq.numel()

        dataset_size = len(train_loader.dataset)

        avg_train_loss = train_loss / dataset_size
        avg_train_crf_loss = train_crf_loss / dataset_size
        avg_train_aux_loss = train_aux_loss / dataset_size
        avg_train_moe_loss = train_moe_loss / dataset_size

        train_acc = correct / max(total, 1)

        current_lr = optimizer.param_groups[0]['lr']

        # Validation stage (whole-well CRF decoding)
        val_eval = detailed_evaluate_model(
            val_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            eval_epoch=epoch
        )
        val_loss = val_eval['avg_loss']
        val_acc = val_eval['overall_accuracy']

        scheduler.step()
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # Logging
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:3d}/{args.epoch:3d} | LR: {current_lr:.6f}")
            print(
                f"Train: total loss {avg_train_loss:.4f} | "
                f"CRF {avg_train_crf_loss:.4f} | "
                f"aux classification {avg_train_aux_loss:.4f} | "
                f"MoE {avg_train_moe_loss:.4f} | "
                f"accuracy {train_acc:.4f}"
            )
            print(
                f"Validation: total loss {val_eval['avg_loss']:.4f} | "
                f"CRF {val_eval['avg_crf_loss']:.4f} | "
                f"aux classification {val_eval['avg_aux_loss']:.4f} | "
                f"MoE {val_eval['avg_moe_loss']:.4f} | "
                f"accuracy {val_eval['overall_accuracy']:.4f} | "
                f"Macro-F1 {val_eval['f1_macro']:.4f}"
            )
            print(
                f"Current progressive sparsity: "
                f"{model.base_model.current_sparse_rate:.4f}"
            )
            if router_stat_batches > 0:
                avg_importance = router_importance_sum / router_stat_batches
                avg_selection = router_selection_sum / router_stat_batches
                avg_entropy = router_entropy_sum / router_stat_batches
                print("Expert soft probabilities:", np.round(avg_importance, 4).tolist())
                print("Expert selection share:", np.round(avg_selection, 4).tolist())
                print(f"Normalised routing entropy: {avg_entropy:.4f}")
            else:
                print("MoE not active yet: still in the warm-up stage")
            print(f"  Early-stopping counter: {early_stop_counter}/{patience} | Best accuracy: {best_val_acc:.4f}")
            print("-" * 80)

        # Save the best model (based on validation Macro-F1)
        val_acc = val_eval["overall_accuracy"]
        val_f1 = val_eval["f1_macro"]

        moe_training_active = (
            epoch + 1 >= args.warm_up_epoch + 1
        )

        if not moe_training_active:
            early_stop_counter = 0
        elif val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(model.state_dict())

            early_stop_counter = 0

            model_save_path = os.path.join(
                args.save_dir,
                args.save_model_name
            )

            torch.save(
                {
                    "epoch": best_epoch,
                    "model_state_dict": best_model_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_f1": float(best_val_f1),
                    "best_val_acc": float(best_val_acc),
                    "args": vars(args),
                    "used_seed": args.random_seed,
                    "class_weights": [float(w) for w in class_weights],
                    "class_order": class_order
                },
                model_save_path
            )

            print(
                f"Saved best model: epoch {best_epoch} | "
                f"Macro-F1 {best_val_f1:.4f} | "
                f"accuracy {best_val_acc:.4f}"
            )
        else:
            early_stop_counter += 1

            if early_stop_counter >= patience:
                print(
                    f"Early stopping triggered: {patience} consecutive epochs "
                    f"without improvement in validation Macro-F1"
                )
                break

    # Post-training
    total_train_time = time.time() - start_time
    print(f"\n=== Training completed ===")
    print(f"Total training time: {total_train_time:.2f} s ({total_train_time / 60:.2f} min)")
    print(f"Random seed used: {args.random_seed}")
    print(f"Best validation accuracy: {best_val_acc:.4f}")

    # Load the best model for the final evaluation
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        final_eval = detailed_evaluate_model(
            val_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            # best_epoch is 1-based
            eval_epoch=best_epoch - 1
        )
        test_eval = detailed_evaluate_model(
            test_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            eval_epoch=best_epoch - 1
        )
        print(f"\n=== Final validation and test accuracy ===")
        print(f"Validation: accuracy {final_eval['overall_accuracy']:.4f} | Macro-F1 {final_eval['f1_macro']:.4f}")
        print(f"Test: accuracy {test_eval['overall_accuracy']:.4f} | Macro-F1 {test_eval['f1_macro']:.4f}")

        # Count sequence errors in the final predictions
        sequence_error_stats = (
            count_sequence_errors(
                final_eval["well_accuracy"],
                class_order
            )
        )
        test_sequence_error_stats = (
            count_sequence_errors(
                test_eval["well_accuracy"],
                class_order
            )
        )

        print("\nValidation sequence error statistics:")
        print(
            f"Total predicted transitions: "
            f"{sequence_error_stats['total_transitions']}"
        )
        print(
            f"Reverse transitions: "
            f"{sequence_error_stats['reverse_transitions']}"
        )
        print(
            f"Layer-skip transitions: "
            f"{sequence_error_stats['skip_transitions']}"
        )
        print(
            f"Total sequence errors: "
            f"{sequence_error_stats['total_sequence_errors']}"
        )
        print(
            f"Sequence error rate: "
            f"{sequence_error_stats['sequence_error_rate']:.4f}"
        )

        print("\nTest sequence error statistics:")
        print(
            f"Total predicted transitions: "
            f"{test_sequence_error_stats['total_transitions']}"
        )
        print(
            f"Reverse transitions: "
            f"{test_sequence_error_stats['reverse_transitions']}"
        )
        print(
            f"Layer-skip transitions: "
            f"{test_sequence_error_stats['skip_transitions']}"
        )
        print(
            f"Total sequence errors: "
            f"{test_sequence_error_stats['total_sequence_errors']}"
        )
        print(
            f"Sequence error rate: "
            f"{test_sequence_error_stats['sequence_error_rate']:.4f}"
        )

        print(f"\n=== Detailed evaluation of the best model ===")
        print(f"📊 Overall metrics:")
        print(f"  Accuracy: {final_eval['overall_accuracy']:.4f}")
        print(f"  Macro precision: {final_eval['precision_macro']:.4f}")
        print(f"  Macro recall: {final_eval['recall_macro']:.4f}")
        print(f"  Macro F1: {final_eval['f1_macro']:.4f}")
        print(f"  Micro precision: {final_eval['precision_micro']:.4f}")
        print(f"  Micro recall: {final_eval['recall_micro']:.4f}")
        print(f"  Micro F1: {final_eval['f1_micro']:.4f}")

        print(f"\n🏭 Per-well validation accuracy:")
        well_acc_df = pd.DataFrame.from_dict(final_eval['well_accuracy'], orient='index')
        well_acc_df = well_acc_df.sort_values('accuracy', ascending=False)
        well_acc_df['accuracy'] = well_acc_df['accuracy'].apply(lambda x: f"{x:.4f}")
        print(well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']])

        print(f"\n📋 Validation classification report:")
        print(final_eval['class_report'])

        print(f"\n🏭 Per-well test accuracy:")
        test_well_acc_df = pd.DataFrame.from_dict(test_eval['well_accuracy'], orient='index')
        test_well_acc_df = test_well_acc_df.sort_values('accuracy', ascending=False)
        test_well_acc_df['accuracy'] = test_well_acc_df['accuracy'].apply(lambda x: f"{x:.4f}")
        print(test_well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']])

        print(f"\n📋 Test classification report:")
        print(test_eval['class_report'])

        # Print per-slice results for every validation well
        print(f"\n=== Detailed validation-sample results (sorted by numeric slice index) ===")
        for well_name, well_info in final_eval['well_accuracy'].items():
            print(f"\n📌 Well: {well_name} (accuracy: {well_info['accuracy']:.4f}, samples: {well_info['sample_count']})")
            print(f"{'Sample ID':<20} {'True label':<15} {'Predicted label':<15} {'Correct'}")
            print("-" * 65)

            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            for _, row in samples_df.iterrows():
                correct_mark = "✅" if row['is_correct'] else "❌"
                print(
                    f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}")

        # Save the detailed text report
        detail_save_path = os.path.join(
            args.save_dir,
            f"detailed_validation_report_"
            f"{args.experiment_name}_"
            f"seed_{args.random_seed}.txt"
        )
        with open(detail_save_path, 'w', encoding='utf-8') as f:
            f.write(f"Experiment name: {args.experiment_name}\n")
            f.write(f"MoE experts: {args.moe_num_experts}\n")
            f.write(f"MoE Top-k: {args.moe_top_k}\n")
            f.write(f"Target sparsity: {args.sparse_rate}\n")
            f.write(f"Warm-up epochs: {args.warm_up_epoch}\n")
            f.write(f"Sparse ramp epochs: {args.sparse_ramp_epoch}\n")
            f.write(f"MoE loss weight: {args.moe_loss_rate}\n")
            f.write(f"Invalid transition penalty: {args.transition_penalty}\n")

            f.write("=== Detailed well-log classification validation report (three-path hybrid model) ===\n")
            f.write(f"Generated at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Random seed used: {args.random_seed}\n")
            f.write(f"Data directory: {args.data_dir}\n")
            f.write(f"Training ratio of remaining wells: {args.train_ratio} | Validation ratio: {1 - args.train_ratio}\n")
            f.write(f"Best-model epoch: {best_epoch}\n")
            f.write(f"Total training time: {total_train_time:.2f} s\n\n")

            f.write("1. Overall validation-set classification metrics\n")
            f.write(f"Accuracy: {final_eval['overall_accuracy']:.4f}\n")
            f.write(f"Macro precision: {final_eval['precision_macro']:.4f}\n")
            f.write(f"Macro recall: {final_eval['recall_macro']:.4f}\n")
            f.write(f"Macro F1: {final_eval['f1_macro']:.4f}\n")
            f.write(f"Micro precision: {final_eval['precision_micro']:.4f}\n")
            f.write(f"Micro recall: {final_eval['recall_micro']:.4f}\n")
            f.write(f"Micro F1: {final_eval['f1_micro']:.4f}\n\n")

            f.write("2. Overall test-set classification metrics\n")
            f.write(f"Accuracy: {test_eval['overall_accuracy']:.4f}\n")
            f.write(f"Macro precision: {test_eval['precision_macro']:.4f}\n")
            f.write(f"Macro recall: {test_eval['recall_macro']:.4f}\n")
            f.write(f"Macro F1: {test_eval['f1_macro']:.4f}\n")
            f.write(f"Micro precision: {test_eval['precision_micro']:.4f}\n")
            f.write(f"Micro recall: {test_eval['recall_micro']:.4f}\n")
            f.write(f"Micro F1: {test_eval['f1_micro']:.4f}\n\n")

            f.write("3. Dataset information\n")
            f.write(f"Training wells: {[well['well_name'] for well in train_wells]}\n")
            f.write(f"Validation wells: {[well['well_name'] for well in val_wells]}\n")
            f.write(f"Test wells: {[well['well_name'] for well in test_wells]}\n")
            f.write(f"Training samples: {len(X_train)} | Validation samples: {len(X_val)} | Test samples: {len(X_test)}\n\n")

            f.write("4. Per-well validation accuracy\n")
            f.write(well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']].to_string())
            f.write("\n\n5. Detailed validation classification report\n")
            f.write(final_eval['class_report'])

            f.write("\n\n6. Per-well test accuracy\n")
            f.write(test_well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']].to_string())
            f.write("\n\n7. Detailed test classification report\n")
            f.write(test_eval['class_report'])

            f.write("\n\n8. Detailed validation-sample results (sorted by numeric slice index)\n")
            for well_name, well_info in final_eval['well_accuracy'].items():
                f.write(f"\n📌 Well: {well_name} (accuracy: {well_info['accuracy']:.4f}, samples: {well_info['sample_count']})\n")
                f.write(f"{'Sample ID':<20} {'True label':<15} {'Predicted label':<15} {'Correct'}\n")
                f.write("-" * 65 + "\n")

                samples_df = well_info['sorted_samples'].copy()
                samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
                samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
                samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

                for _, row in samples_df.iterrows():
                    correct_mark = "✅" if row['is_correct'] else "❌"
                    f.write(
                        f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}\n")

            f.write("\n\n9. Detailed test-sample results (sorted by numeric slice index)\n")
            for well_name, well_info in test_eval['well_accuracy'].items():
                f.write(f"\n📌 Well: {well_name} (accuracy: {well_info['accuracy']:.4f}, samples: {well_info['sample_count']})\n")
                f.write(f"{'Sample ID':<20} {'True label':<15} {'Predicted label':<15} {'Correct'}\n")
                f.write("-" * 65 + "\n")

                samples_df = well_info['sorted_samples'].copy()
                samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
                samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
                samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

                for _, row in samples_df.iterrows():
                    correct_mark = "✅" if row['is_correct'] else "❌"
                    f.write(
                        f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}\n")

            # Section 10: validation-set sequence error statistics
            f.write("\n\n10. Validation sequence error statistics\n")
            f.write(f"Total predicted transitions: {sequence_error_stats['total_transitions']}\n")
            f.write(f"Reverse transitions: {sequence_error_stats['reverse_transitions']}\n")
            f.write(f"Layer-skip transitions: {sequence_error_stats['skip_transitions']}\n")
            f.write(f"Total sequence errors: {sequence_error_stats['total_sequence_errors']}\n")
            f.write(f"Sequence error rate: {sequence_error_stats['sequence_error_rate']:.4f}\n")
            f.write("\nSequence error details:\n")
            f.write(
                f"{'Well':<15}"
                f"{'Previous sample':<25}"
                f"{'Next sample':<25}"
                f"{'Type':<12}\n"
            )
            f.write("-" * 85 + "\n")

            for item in sequence_error_stats["details"]:
                from_name = label_encoder.inverse_transform(
                    [item["from_tag"]]
                )[0]
                to_name = label_encoder.inverse_transform(
                    [item["to_tag"]]
                )[0]
                transition_text = (
                    f"{from_name}->{to_name}"
                )
                f.write(
                    f"{item['well_name']:<15}"
                    f"{item['from_sample']:<25}"
                    f"{item['to_sample']:<25}"
                    f"{item['type']:<12}"
                    f"{transition_text}\n"
                )

            f.write("\n\n11. Test sequence error statistics\n")
            f.write(f"Total predicted transitions: {test_sequence_error_stats['total_transitions']}\n")
            f.write(f"Reverse transitions: {test_sequence_error_stats['reverse_transitions']}\n")
            f.write(f"Layer-skip transitions: {test_sequence_error_stats['skip_transitions']}\n")
            f.write(f"Total sequence errors: {test_sequence_error_stats['total_sequence_errors']}\n")
            f.write(f"Sequence error rate: {test_sequence_error_stats['sequence_error_rate']:.4f}\n")
            f.write("\nSequence error details:\n")
            f.write(
                f"{'Well':<15}"
                f"{'Previous sample':<25}"
                f"{'Next sample':<25}"
                f"{'Type':<12}\n"
            )
            f.write("-" * 85 + "\n")

            for item in test_sequence_error_stats["details"]:
                from_name = label_encoder.inverse_transform(
                    [item["from_tag"]]
                )[0]
                to_name = label_encoder.inverse_transform(
                    [item["to_tag"]]
                )[0]
                transition_text = (
                    f"{from_name}->{to_name}"
                )
                f.write(
                    f"{item['well_name']:<15}"
                    f"{item['from_sample']:<25}"
                    f"{item['to_sample']:<25}"
                    f"{item['type']:<12}"
                    f"{transition_text}\n"
                )

        print(f"\n📄 Detailed report saved to: {detail_save_path}")

        # Save validation confusion matrix
        conf_matrix = final_eval['confusion_matrix']
        conf_df = pd.DataFrame(conf_matrix, index=label_encoder.classes_, columns=label_encoder.classes_)
        conf_csv_path = os.path.join(args.save_dir, f'confusion_matrix_seed_{args.random_seed}.csv')
        conf_df.to_csv(conf_csv_path, encoding='utf-8-sig')
        print(f"📊 Confusion matrix saved to: {conf_csv_path}")

        # Plot validation confusion matrix
        plt.figure(figsize=(12, 10))
        plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(f'Confusion matrix (seed: {args.random_seed} | three-path hybrid model)', fontsize=14)
        plt.colorbar()
        tick_marks = np.arange(len(label_encoder.classes_))
        plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha='right')
        plt.yticks(tick_marks, label_encoder.classes_)

        thresh = conf_matrix.max() / 2.
        for i in range(conf_matrix.shape[0]):
            for j in range(conf_matrix.shape[1]):
                plt.text(j, i, format(conf_matrix[i, j], 'd'),
                         horizontalalignment="center",
                         color="white" if conf_matrix[i, j] > thresh else "black")

        plt.ylabel('True label', fontsize=12)
        plt.xlabel('Predicted label', fontsize=12)
        plt.tight_layout()
        conf_plot_path = os.path.join(args.save_dir, f'confusion_matrix_seed_{args.random_seed}.png')
        plt.savefig(conf_plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Confusion-matrix figure saved to: {conf_plot_path}")

        # Save test confusion matrix
        test_conf_matrix = test_eval['confusion_matrix']
        test_conf_df = pd.DataFrame(test_conf_matrix, index=label_encoder.classes_, columns=label_encoder.classes_)
        test_conf_csv_path = os.path.join(args.save_dir, f'test_confusion_matrix_seed_{args.random_seed}.csv')
        test_conf_df.to_csv(test_conf_csv_path, encoding='utf-8-sig')
        print(f"📊 Test confusion matrix saved to: {test_conf_csv_path}")

        # Plot test confusion matrix
        plt.figure(figsize=(12, 10))
        plt.imshow(test_conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(f'Test confusion matrix (seed: {args.random_seed} | three-path hybrid model)', fontsize=14)
        plt.colorbar()
        tick_marks = np.arange(len(label_encoder.classes_))
        plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha='right')
        plt.yticks(tick_marks, label_encoder.classes_)

        test_thresh = test_conf_matrix.max() / 2.
        for i in range(test_conf_matrix.shape[0]):
            for j in range(test_conf_matrix.shape[1]):
                plt.text(j, i, format(test_conf_matrix[i, j], 'd'),
                         horizontalalignment="center",
                         color="white" if test_conf_matrix[i, j] > test_thresh else "black")

        plt.ylabel('True label', fontsize=12)
        plt.xlabel('Predicted label', fontsize=12)
        plt.tight_layout()
        test_conf_plot_path = os.path.join(args.save_dir, f'test_confusion_matrix_seed_{args.random_seed}.png')
        plt.savefig(test_conf_plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Test confusion-matrix figure saved to: {test_conf_plot_path}")

        # Save the training result CSV
        csv_save_path = os.path.join(args.save_dir, args.save_csv_name + '.csv')
        save_cls_result(args, final_eval, total_train_time, label_encoder, csv_save_path, args.random_seed)

        # Training history figure
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
        epochs = len(history['train_loss'])

        ax1.plot(range(1, epochs + 1), history['train_loss'], label='Training loss', color='#1f77b4')
        ax1.plot(range(1, epochs + 1), history['val_loss'], label='Validation loss', color='#ff7f0e')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title(f'Loss curves (seed: {args.random_seed} | three-path hybrid model)')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(range(1, epochs + 1), history['train_acc'], label='Training accuracy', color='#1f77b4')
        ax2.plot(range(1, epochs + 1), history['val_acc'], label='Validation accuracy', color='#ff7f0e')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title(f'Accuracy curves (best validation accuracy: {best_val_acc:.4f})')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(range(1, epochs + 1), history['lr'], label='Learning rate', color='#2ca02c')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning rate')
        ax3.set_title('Learning-rate schedule (cosine annealing)')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.axis('off')
        config_text = f"""
        Model configuration (seed: {args.random_seed} | three-path hybrid model)
        ========================
        Data directory: {args.data_dir}
        Training wells: {len(train_wells)} | Validation wells: {len(val_wells)} | Test wells: {len(test_wells)}
        Data dimensions: {args.num_channels} channels x {args.seq_len} depth samples
        Model size: {total_params / 1e6:.2f}M trainable parameters
        Best results: validation accuracy={best_val_acc:.4f} | test accuracy={test_eval['overall_accuracy']:.4f}
        Training time: {total_train_time:.2f} s
        Architecture: SoftShape MoE + class-specific Transformer + generic Transformer
        """
        ax4.text(0.1, 0.9, config_text, transform=ax4.transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plot_save_path = os.path.join(
            args.save_dir,
            f"training_history_"
            f"{args.experiment_name}_"
            f"seed_{args.random_seed}.png"
        )
        plt.tight_layout()
        plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📈 Training-history figure saved to: {plot_save_path}")

        # Save per-well validation result tables
        print(f"\n=== Saving per-well validation result tables ===")
        for well_name, well_info in final_eval['well_accuracy'].items():
            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            save_cols = ['sample_id', 'true_label_name', 'pred_label_name', 'is_correct']
            well_result_df = samples_df[save_cols]

            well_result_path = os.path.join(args.save_dir,
                                            f'well_{well_name}_validation_result_seed_{args.random_seed}.csv')
            well_result_df.to_csv(well_result_path, index=False, encoding='utf-8-sig')
            print(f"📋 Validation results for well {well_name} saved to: {well_result_path}")

        # Save per-well test result tables
        print(f"\n=== Saving per-well test result tables ===")
        for well_name, well_info in test_eval['well_accuracy'].items():
            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            save_cols = ['sample_id', 'true_label_name', 'pred_label_name', 'is_correct']
            well_result_df = samples_df[save_cols]

            well_result_path = os.path.join(args.save_dir,
                                            f'well_{well_name}_test_result_seed_{args.random_seed}.csv')
            well_result_df.to_csv(well_result_path, index=False, encoding='utf-8-sig')
            print(f"📋 Test results for well {well_name} saved to: {well_result_path}")

    print("\n=== All steps completed ===")
